<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_03_statistical_experiments_and_significance_testing/exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 3 folder
%cd chapter_03_statistical_experiments_and_significance_testing

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (125/125), done.
Receiving objects: 100% (164/164), 568.82 KiB | 1.48 MiB/s, done.
remote: Total 164 (delta 99), reused 70 (delta 37), pack-reused 0 (from 0)
Resolving deltas: 100% (99/99), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_03_statistical_experiments_and_significance_testing


# Chapter 03 Exercises: Statistical Experiments and Significance Testing

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 3: Statistical Experiments and Significance Testing

## Goals

In this notebook, I will practise:
- A/B testing
- Hypothesis testing
- Null hypothesis (H₀) and alternative hypothesis (H₁)
- Statistical significance
- P-values and their interpretation
- Permutation tests
- T-tests
- Type I and Type II errors
- Statistical power and sample size
- Multiple testing and corrections
- ANOVA
- Chi-Square tests
- Multi-Arm Bandits

---

## 📋 Instructions

- Attempt each exercise before viewing the solution
- Use `utils/notebook_setup.py` for consistent imports when available
- Focus on understanding the logic of resampling
- Solutions are provided in collapsed sections—expand only after attempting the problem

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats
    from scipy.stats import ttest_ind, f_oneway, chi2_contingency, fisher_exact
    from statsmodels.stats.multitest import multipletests
    from statsmodels.stats.proportion import proportions_ztest, proportion_confint
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

# Helper to simulate book datasets
def load_or_simulate_data():
    # 1. Web Stickiness (Continuous)
    time_a = np.maximum(np.random.normal(120, 40, 21), 10)
    time_b = np.maximum(np.random.normal(145, 45, 15), 10)
    df_sessions = pd.DataFrame({
        'time': np.concatenate([time_a, time_b]),
        'page': ['Page_A']*21 + ['Page_B']*15
    })
    
    # 2. E-commerce Price Test (Binary)
    conv_a = np.random.binomial(1, 0.045, 23739)
    conv_b = np.random.binomial(1, 0.055, 22588)
    df_price = pd.DataFrame({
        'converted': np.concatenate([conv_a, conv_b]),
        'group': ['Price_A']*23739 + ['Price_B']*22588
    })
    
    # 3. Four Web Pages (ANOVA)
    df_four = pd.DataFrame({
        'time': np.concatenate([
            np.random.normal(110, 30, 50),
            np.random.normal(125, 35, 50),
            np.random.normal(115, 32, 50),
            np.random.normal(140, 40, 50)
        ]),
        'page': ['P1']*50 + ['P2']*50 + ['P3']*50 + ['P4']*50
    })
    
    return df_sessions, df_price, df_four

df_sessions, df_price, df_four = load_or_simulate_data()

# Simple A/B test data
group_a = np.random.binomial(1, 0.12, 1000)
group_b = np.random.binomial(1, 0.14, 1000)

print("Datasets loaded/simulated successfully.")
print(f"Sessions: {df_sessions.shape}, Price Test: {df_price.shape}, Four Pages: {df_four.shape}")
```
</details>


---

# Exercise 1: A/B Testing Basics

## Scenario

You are testing a new airline booking page.

- **Group A**: Current booking page
- **Group B**: New booking page

Goal: Determine whether new page improves conversion.

<details>
<summary>Click to view solution</summary>

```python
print("Group A Conversion Rate:", f"{group_a.mean():.3f}")
print("Group B Conversion Rate:", f"{group_b.mean():.3f}")

comparison = pd.DataFrame({
    "Group": ["A", "B"],
    "Conversion Rate": [group_a.mean(), group_b.mean()]
})
print(comparison)

plt.figure(figsize=(7, 5))
sns.barplot(data=comparison, x="Group", y="Conversion Rate")
plt.title("A/B Test Conversion Rates")
plt.show()
```
</details>

## Reflection

Questions:
1. Which group performed better?
2. Is the difference large?
3. Could randomness explain the result?


---

# Exercise 2: Null vs Alternative Hypothesis

## Task

Define:
- **H₀ (Null Hypothesis)**: What assumption are we starting with?
- **H₁ (Alternative Hypothesis)**: What are we trying to prove?

### Write your answers here

#### H₀
No difference exists between groups. Any observed difference is due to random chance.

#### H₁
New website improves conversion. There is a real difference between groups.



---

# Exercise 3: Statistical Significance

## Task

Interpret the following:

- **Case 1**: p = 0.03
- **Case 2**: p = 0.12

Questions:
1. Which result is statistically significant?
2. Which one rejects H₀?
3. Does significant mean useful?

### Write your interpretation below

Case 1 (p=0.03) is statistically significant at α=0.05 and rejects H₀. Case 2 (p=0.12) is not statistically significant. However, statistical significance does not guarantee practical usefulness—a tiny effect can be significant with large samples.


---

# Exercise 4: P-value Calculation

## Task

Run an independent t-test. Is the difference statistically significant?

<details>
<summary>Click to view solution</summary>

```python
t_stat, p_value = stats.ttest_ind(group_a, group_b)

print("T-statistic:", t_stat)
print("P-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("Reject Null Hypothesis - Result is statistically significant")
else:
    print("Fail to Reject Null Hypothesis")
```
</details>

## Reflection

Questions:
1. Is p-value below 0.05?
2. Should H₀ be rejected?
3. Does this prove the new version is better?



---

# Exercise 5: Randomness Experiment

## Question

Can random chance create differences? Create two identical groups and test.

<details>
<summary>Click to view solution</summary>

```python
random_a = np.random.normal(50, 10, 1000)
random_b = np.random.normal(50, 10, 1000)

t_stat, p_value = stats.ttest_ind(random_a, random_b)

print("P-value:", p_value)
print("Both groups drawn from same distribution - any 'difference' is purely random.")
print("We should NOT reject H₀ here (p should typically be > 0.05)")
```
</details>

## Reflection

Questions:
1. Were groups actually different?
2. Did randomness still create variation?
3. Why do we need significance testing?



---

# Exercise 6: Effect Size Experiment

## Task

Change effect from 0.14 to 0.20 and observe how p-value changes.

<details>
<summary>Click to view solution</summary>

```python
group_c = np.random.binomial(1, 0.20, 1000)

t_stat, p_value = stats.ttest_ind(group_a, group_c)

print("P-value with larger effect (0.12 vs 0.20):", p_value)
print("The p-value should be much smaller because the effect is stronger.")
```
</details>

## Reflection

Questions:
1. Did p-value become smaller?
2. Why?
3. Why do strong signals matter?


---

# Exercise 7: Business Interpretation

## Scenario

Conversion improves from 12% to 12.2%. P-value = 0.001.

Questions:
1. Is result statistically significant?
2. Is improvement meaningful?
3. Would you implement this change?

### Write your reasoning below

The result is statistically significant (p < 0.05), but the 0.2% absolute improvement may not be practically meaningful. Consider the cost of implementation, engineering effort, and whether the improvement justifies the investment. Statistical significance alone should not drive business decisions.


---

# Exercise 8: Permutation Test (Continuous Data)

## Context

You are testing two web page layouts (Page A: 21 sessions, Page B: 15 sessions) to see which keeps users engaged longer.

## Task
1. Calculate the observed difference in mean session times (Page B - Page A)
2. Write a permutation test function
3. Calculate the p-value
4. Visualize the null distribution

<details>
<summary>Click to view solution</summary>

```python
# 1. Observed difference
group_a_sessions = df_sessions[df_sessions['page'] == 'Page_A']['time'].values
group_b_sessions = df_sessions[df_sessions['page'] == 'Page_B']['time'].values
obs_diff = np.mean(group_b_sessions) - np.mean(group_a_sessions)
print(f"Observed Difference (B - A): {obs_diff:.2f} seconds")

# 2. Permutation Test Function
def permutation_test(g_a, g_b, n_permutations=3000, seed=42):
    np.random.seed(seed)
    pooled = np.concatenate([g_a, g_b])
    n_a = len(g_a)
    perm_diffs = np.zeros(n_permutations)
    
    for i in range(n_permutations):
        np.random.shuffle(pooled)
        perm_a = pooled[:n_a]
        perm_b = pooled[n_a:]
        perm_diffs[i] = np.mean(perm_b) - np.mean(perm_a)
        
    return perm_diffs

# 3. Run and calculate p-value
perm_diffs = permutation_test(group_a_sessions, group_b_sessions)
p_value_one_sided = np.mean(perm_diffs >= obs_diff)
print(f"Permutation p-value (one-sided): {p_value_one_sided:.4f}")

# 4. Visualization
plt.figure(figsize=(8, 5))
plt.hist(perm_diffs, bins=30, color='skyblue', edgecolor='black', alpha=0.7, label='Null Distribution')
plt.axvline(obs_diff, color='red', linestyle='--', linewidth=2, label=f'Observed Diff ({obs_diff:.2f})')
plt.xlabel('Difference in Means (Page B - Page A)')
plt.ylabel('Frequency')
plt.title(f'Permutation Test Null Distribution (p = {p_value_one_sided:.3f})')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Simple Permutation Test with A/B Data

<details>
<summary>Click to view solution</summary>

```python
observed_difference = group_b.mean() - group_a.mean()
print("Observed Difference:", observed_difference)

combined = np.concatenate([group_a, group_b])
permutation_differences = []

for _ in range(1000):
    shuffled = np.random.permutation(combined)
    new_a = shuffled[:1000]
    new_b = shuffled[1000:]
    diff = new_b.mean() - new_a.mean()
    permutation_differences.append(diff)

plt.figure(figsize=(10, 6))
sns.histplot(permutation_differences, bins=30, kde=True)
plt.axvline(observed_difference, linestyle="--", label="Observed Difference")
plt.legend()
plt.title("Permutation Test Distribution")
plt.show()

p_value_perm = np.mean(np.abs(permutation_differences) >= np.abs(observed_difference))
print("Permutation P-value:", p_value_perm)
```
</details>

## Reflection

Questions:
1. Was observed difference unusual?
2. Why does shuffling matter?
3. Why are permutation tests intuitive?



---

# Exercise 9: Resampling for Proportions (Binary Data)

## Context

An e-commerce site tests a new price against the standard price:
- Price A: 200 conversions out of 23,739 visits
- Price B: 182 conversions out of 22,588 visits

Price A actually has a slightly higher conversion rate. Is this difference statistically significant?

## Task
1. Create a "box" of 1s and 0s representing the pooled null hypothesis
2. Perform a permutation test for the difference in proportions
3. Compare to the analytical p-value from Chi-Square test

<details>
<summary>Click to view solution</summary>

```python
# Observed data
conv_a, n_a = 200, 23739
conv_b, n_b = 182, 22588
obs_pct_diff = 100 * (conv_a/n_a - conv_b/n_b)
print(f"Observed difference in conversion % (A - B): {obs_pct_diff:.4f}%")

# 1. Create the pooled box
box = np.concatenate([np.ones(conv_a + conv_b), np.zeros(n_a + n_b - (conv_a + conv_b))])

# 2. Permutation Test for Proportions
np.random.seed(42)
n_perm = 2000
perm_pct_diffs = np.zeros(n_perm)

for i in range(n_perm):
    np.random.shuffle(box)
    perm_a = box[:n_a]
    perm_b = box[n_a:]
    perm_pct_diffs[i] = 100 * (np.mean(perm_a) - np.mean(perm_b))

p_val_perm = np.mean(perm_pct_diffs >= obs_pct_diff)
print(f"Resampled p-value: {p_val_perm:.4f}")

# 3. Analytical Chi-Square Test
contingency = np.array([[conv_a, n_a - conv_a],
                        [conv_b, n_b - conv_b]])
chi2, p_val_chi, dof, expected = chi2_contingency(contingency)
print(f"Chi-Square p-value: {p_val_chi:.4f}")

# Visualization
plt.figure(figsize=(8, 5))
plt.hist(perm_pct_diffs, bins=25, color='lightgreen', edgecolor='black', alpha=0.7)
plt.axvline(obs_pct_diff, color='red', linestyle='--', linewidth=2, label=f'Observed ({obs_pct_diff:.3f}%)')
plt.xlabel('Difference in Conversion % (A - B)')
plt.title('Permutation Distribution of Conversion Rates')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("\nConclusion: The observed difference is well within the range of chance variation.")
```
</details>


---

# Exercise 10: Type I Error (False Positive)

## Question

Can significance appear even when nothing changes? Simulate many experiments where no real difference exists.

<details>
<summary>Click to view solution</summary>

```python
false_positives = 0

for _ in range(1000):
    a = np.random.normal(50, 10, 100)
    b = np.random.normal(50, 10, 100)
    _, p = stats.ttest_ind(a, b)
    if p < 0.05:
        false_positives += 1

print("False Positive Rate:", false_positives / 1000)
print("Expected: ~0.05 (5% of tests falsely reject H₀)")
```
</details>

## Reflection

Questions:
1. Was error rate near 5%?
2. Why do false positives happen?
3. Why can companies fool themselves?



---

# Exercise 11: Type II Error (False Negative)

## Task

Create a small sample with a real difference. Can we miss a real effect?

<details>
<summary>Click to view solution</summary>

```python
small_a = np.random.normal(50, 10, 10)
small_b = np.random.normal(53, 10, 10)

_, p_small = stats.ttest_ind(small_a, small_b)

print("P-value with small sample:", p_small)
print("Note: Real effect exists (53 vs 50) but may not be detected with small n")
print("This demonstrates Type II error - failing to detect a real effect.")
```
</details>

## Reflection

Questions:
1. Was result significant?
2. Could real effect still exist?
3. Why are small samples risky?


---

# Exercise 12: Statistical Power

## Question

What happens when sample size increases? Compare p-values across different sample sizes.

<details>
<summary>Click to view solution</summary>

```python
sample_sizes = [10, 50, 100, 500]
p_values = []

for n in sample_sizes:
    a = np.random.normal(50, 10, n)
    b = np.random.normal(53, 10, n)
    _, p = stats.ttest_ind(a, b)
    p_values.append(p)

power_df = pd.DataFrame({"Sample Size": sample_sizes, "P-value": p_values})
print(power_df)

plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, p_values, marker="o")
plt.axhline(0.05, linestyle="--", label="Threshold")
plt.legend()
plt.title("Sample Size vs P-value")
plt.xlabel("Sample Size")
plt.ylabel("P-value")
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Reflection

Questions:
1. What happened as sample size increased?
2. Why does more data improve detection?
3. Why can tiny effects become significant?


---

# Exercise 13: Power and Sample Size via Simulation

## Context

Before launching an A/B test, you need to know how many users to collect. You want to detect an absolute lift in conversion from 10% to 12% (Effect Size = 2%).

## Task
1. Write a function `simulate_power` that generates binary data and runs a Chi-Square test
2. Test sample sizes: [500, 1000, 2000, 4000, 6000, 8000]
3. Plot the power curve and determine required sample size for ~80% power

<details>
<summary>Click to view solution</summary>

```python
def simulate_power(n_per_group, p_control, p_treatment, alpha=0.05, n_simulations=500):
    significant_count = 0
    for _ in range(n_simulations):
        conv_a = np.random.binomial(1, p_control, n_per_group)
        conv_b = np.random.binomial(1, p_treatment, n_per_group)
        
        table = np.array([
            [sum(conv_a), n_per_group - sum(conv_a)],
            [sum(conv_b), n_per_group - sum(conv_b)]
        ])
        
        _, p_val, _, _ = chi2_contingency(table)
        if p_val < alpha:
            significant_count += 1
            
    return significant_count / n_simulations

# Run simulations
sample_sizes_power = [500, 1000, 2000, 4000, 6000, 8000]
powers = []

print("Simulating Power (detecting 10% -> 12% lift)...")
for n in sample_sizes_power:
    power = simulate_power(n, p_control=0.10, p_treatment=0.12, n_simulations=300)
    powers.append(power)
    print(f"  n={n:>5,} per group -> Power: {power:.1%}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(sample_sizes_power, powers, 'o-', linewidth=2, color='purple')
plt.axhline(0.80, color='red', linestyle='--', label='80% Power Target')
plt.xlabel('Sample Size per Group')
plt.ylabel('Statistical Power')
plt.title('Power Curve: Detecting 2% Absolute Lift in Conversion')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>


---

# Exercise 14: Multiple Testing Problem

## Task

Run 100 experiments where nothing changes. How many significant results appear?

<details>
<summary>Click to view solution</summary>

```python
significant_results = 0

for _ in range(100):
    a = np.random.normal(50, 10, 100)
    b = np.random.normal(50, 10, 100)
    _, p = stats.ttest_ind(a, b)
    if p < 0.05:
        significant_results += 1

print("Significant Results:", significant_results)
print("Expected ~5 by chance alone at α=0.05")
```
</details>

## Reflection

Questions:
1. Did significant results appear?
2. Why is this dangerous?
3. What is p-hacking?



---

# Exercise 15: Multiple Testing and False Discoveries

## Context

In ML feature selection, you might test 100 features to see if they correlate with your target. If you use α = 0.05, you expect 5 false positives purely by chance.

## Task
1. Generate 100 p-values: 90 from uniform (H₀ true) and 10 from Beta(1, 20) (H₁ true)
2. Count significant at α = 0.05 without correction
3. Apply Bonferroni and FDR corrections
4. Compare true positives and false positives retained by each method

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# 1. Generate p-values
pvals_null = np.random.uniform(0, 1, 90)
pvals_alt = np.random.beta(1, 20, 10)
pvals = np.concatenate([pvals_null, pvals_alt])
is_true_effect = np.array([False]*90 + [True]*10)

# 2. Uncorrected
uncorrected_sig = pvals < 0.05
print(f"Uncorrected: {uncorrected_sig.sum()} significant")
print(f"  - True Positives: {(uncorrected_sig & is_true_effect).sum()}")
print(f"  - False Positives: {(uncorrected_sig & ~is_true_effect).sum()}")

# 3. Corrections
reject_bonf, pvals_bonf, _, _ = multipletests(pvals, alpha=0.05, method='bonferroni')
reject_fdr, pvals_fdr, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

print(f"\nBonferroni: {reject_bonf.sum()} significant")
print(f"  - True Positives: {(reject_bonf & is_true_effect).sum()}")
print(f"  - False Positives: {(reject_bonf & ~is_true_effect).sum()}")

print(f"\nBenjamini-Hochberg (FDR): {reject_fdr.sum()} significant")
print(f"  - True Positives: {(reject_fdr & is_true_effect).sum()}")
print(f"  - False Positives: {(reject_fdr & ~is_true_effect).sum()}")

print("\nML Relevance Insight:")
print("Bonferroni is too strict for high-dimensional data (kills true positives).")
print("FDR (Benjamini-Hochberg) is the standard in Data Science because it controls")
print("the proportion of false discoveries, allowing more true signals to be found.")
```
</details>


---

# Exercise 16: ANOVA (Comparing Multiple Groups)

## Context

You are testing 4 different web page designs (P1, P2, P3, P4) to see if they result in different average session times.

## Task
1. Visualize the data using side-by-side boxplots
2. Run a One-Way ANOVA
3. If significant, run a permutation-based ANOVA to verify

<details>
<summary>Click to view solution</summary>

```python
# 1. Visualization
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_four, x='page', y='time', palette='Set2')
sns.stripplot(data=df_four, x='page', y='time', color='black', alpha=0.3, jitter=True, size=3)
plt.title('Session Times Across 4 Web Pages')
plt.ylabel('Time (seconds)')
plt.grid(axis='y', alpha=0.3)
plt.show()

# 2. Classical One-Way ANOVA
groups = [group['time'].values for _, group in df_four.groupby('page')]
f_stat, p_val_anova = f_oneway(*groups)
print(f"Classical ANOVA: F-stat = {f_stat:.3f}, p-value = {p_val_anova:.4f}")

# 3. Permutation ANOVA
np.random.seed(42)
observed_variance = np.var([np.mean(g) for g in groups])

pooled = df_four['time'].values
n_per_group = 50
perm_variances = np.zeros(1000)

for i in range(1000):
    np.random.shuffle(pooled)
    perm_groups = [pooled[i*n_per_group : (i+1)*n_per_group] for i in range(4)]
    perm_variances[i] = np.var([np.mean(g) for g in perm_groups])

p_val_perm_anova = np.mean(perm_variances >= observed_variance)
print(f"Permutation ANOVA p-value: {p_val_perm_anova:.4f}")

if p_val_anova < 0.05:
    print("Conclusion: At least one page is significantly different.")
    print("Next Step: Use post-hoc tests (like Tukey's HSD) to find exactly WHICH pages differ.")
```
</details>


---

# Exercise 17: Business Significance vs Statistical Significance

## Scenario

Conversion improves from 12.0% to 12.1%. P-value = 0.0001.

Questions:
1. Is it statistically significant?
2. Is it practically useful?
3. Would business implement it?

### Write your reasoning below

The result is statistically significant (p << 0.05), but the 0.1% absolute improvement may not justify implementation costs. With large sample sizes, even tiny effects become statistically significant. Business decisions should consider practical significance, implementation costs, and impact on other metrics—not just p-values.



---

# Exercise 18: Multi-Arm Bandit (Epsilon-Greedy)

## Context

Standard A/B testing wastes traffic on inferior variants. Multi-Arm Bandits dynamically allocate traffic to the best performer.

## Task
1. Implement an `EpsilonGreedyBandit` class
2. Simulate 3 ad variants with true CTRs of 2%, 4%, and 8%
3. Run the bandit for 2,000 steps with ε = 0.1
4. Plot the cumulative average reward over time

<details>
<summary>Click to view solution</summary>

```python
class EpsilonGreedyBandit:
    def __init__(self, n_arms, epsilon=0.1):
        self.epsilon = epsilon
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        
    def select_arm(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(len(self.values))
        return np.argmax(self.values)
        
    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n

# Simulation
np.random.seed(42)
true_rates = [0.02, 0.04, 0.08]
n_steps = 2000
bandit = EpsilonGreedyBandit(n_arms=3, epsilon=0.1)

rewards_history = []
cumulative_avg = []

for step in range(n_steps):
    arm = bandit.select_arm()
    reward = np.random.binomial(1, true_rates[arm])
    bandit.update(arm, reward)
    rewards_history.append(reward)
    cumulative_avg.append(np.mean(rewards_history))

# Results
print("Final Estimated CTRs:")
for i, val in enumerate(bandit.values):
    print(f"  Arm {i}: Estimated {val:.3f} (True: {true_rates[i]}) | Pulled {int(bandit.counts[i])} times")

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(cumulative_avg, label='Epsilon-Greedy Bandit', color='blue')
plt.axhline(max(true_rates), color='red', linestyle='--', label='Optimal Arm True CTR (0.08)')
plt.axhline(np.mean(true_rates), color='gray', linestyle=':', label='Random Guess Average')
plt.xlabel('Steps (Users)')
plt.ylabel('Cumulative Conversion Rate')
plt.title('Epsilon-Greedy Bandit Learning Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>


---

# Exercise 19: Synthesis & Critical Thinking

**Answer in your own words:**

### 1. The P-Value Misconception

A product manager sees an A/B test with a p-value of 0.03 and says, "This means there is a 97% chance the new feature is better." Why is this statistically incorrect? How would you explain the true meaning of the p-value?

<details>
<summary>Click to view solution</summary>

```python
answer_1 = """
The p-value is NOT the probability that the null hypothesis is false,
nor the probability that the alternative is true.

It is the probability of observing a result as extreme as, or more extreme than,
the actual result, ASSUMING the null hypothesis (no difference) is true.

A p-value of 0.03 means: "If the feature had zero effect, we'd only see a lift
this big 3% of the time due to random chance."

It does NOT mean there's a 97% chance the feature works.
"""
print(answer_1)
```
</details>

### 2. Practical vs. Statistical Significance

You run an A/B test with 5 million users. The new checkout button increases conversion from 4.00% to 4.02%. The p-value is 0.001. Should you implement the change? What other factors must you consider?

<details>
<summary>Click to view solution</summary>

```python
answer_2 = """
With N=5,000,000, the test is heavily overpowered. A 0.02% absolute lift is
statistically significant, but is it PRACTICALLY significant?

Consider:
- Engineering/design cost of changing the button
- Actual projected revenue impact of a 0.02% lift
- Whether the new button negatively impacts other metrics
  (page load time, long-term retention, user satisfaction)
- Opportunity cost: Could engineering time be better spent elsewhere?

Statistical significance ≠ Business significance
"""
print(answer_2)
```
</details>

### 3. Bandits vs. A/B Tests

In what business scenario would a Multi-Arm Bandit be vastly superior to a traditional A/B test? In what scenario might a Bandit be dangerous or inappropriate?

<details>
<summary>Click to view solution</summary>

```python
answer_3 = """
Superior for Bandits:
- Short-lived campaigns (e.g., Black Friday headlines) where waiting for
  an A/B test to finish means missing the revenue window
- Situations where you want to minimize regret (showing bad variants)
- When you need to continuously adapt to changing conditions

Dangerous for Bandits:
- When there are strong time-based confounders (day-of-week effects)
- If Arm A is tested mostly on weekdays and Arm B on weekends,
  the bandit might falsely converge on the wrong arm
- Standard A/B tests with simultaneous randomization handle temporal
  confounders better
- When you need clear, interpretable statistical conclusions for stakeholders
"""
print(answer_3)
```
</details>



---

# ML Relevance Exercise

## Question

Why does Chapter 3 matter in Machine Learning?

Think about:
- Model comparison
- A/B testing
- Significance of improvements
- Randomness
- Overfitting
- Unreliable gains

---

# Mini Exercise

## Question

Which matters more: Statistical significance or Business significance? Explain with an example.

---

# Interview Questions

1. **What is hypothesis testing?**
2. **What is null hypothesis?**
3. **What is p-value?**
4. **Difference between statistical and business significance?**
5. **What is Type I error?**
6. **What is Type II error?**
7. **What is statistical power?**
8. **Why are permutation tests useful?**
9. **What is p-hacking?**
10. **Why can multiple testing be dangerous?**
11. **How do you correct for multiple testing?**
12. **What is the difference between ANOVA and t-test?**
13. **When would you use a Chi-Square test?**
14. **What is a Multi-Arm Bandit and when would you use it?**

---

# Challenge Problems

1. Build A/B test simulator where effect size changes and observe p-value changes
2. Simulate 1000 false positives and calculate error rate
3. Compare small sample vs large sample for significance detection
4. Create permutation test vs t-test comparison
5. Simulate multiple testing problem with 500 experiments
6. Implement a more advanced bandit algorithm (UCB or Thompson Sampling)
7. Simulate power curves for different effect sizes

---

# Progress Checklist

- [ ] Practised A/B testing
- [ ] Understood H₀ vs H₁
- [ ] Interpreted p-values
- [ ] Used t-tests
- [ ] Built permutation tests for continuous data
- [ ] Performed permutation tests for proportions
- [ ] Compared permutation vs Chi-Square tests
- [ ] Understood Type I error
- [ ] Understood Type II error
- [ ] Learned statistical power
- [ ] Simulated power analysis
- [ ] Explored multiple testing
- [ ] Applied Bonferroni and FDR corrections
- [ ] Performed ANOVA with permutation verification
- [ ] Implemented Epsilon-Greedy Bandit
- [ ] Completed critical thinking questions
- [ ] Connected concepts to ML
- [ ] Ready for Chapter 4

---

> **Pro Tip**: The `permutation_test` and `simulate_power` functions you wrote here are incredibly reusable. Save them in your `utils/stats_helpers.py` file so you can import them in future ML projects when you need to evaluate if Model A is truly better than Model B!
